# Policy DSL and Petri Net Regime Sequencing

Demonstrates the declarative YAML policy engine and Petri net FSM
for multi-phase protocol control.

In [ ]:
import numpy as np

from scpn_phase_orchestrator.binding.types import BoundaryDef
from scpn_phase_orchestrator.monitor.boundaries import BoundaryObserver
from scpn_phase_orchestrator.supervisor.events import EventBus
from scpn_phase_orchestrator.supervisor.policy import SupervisorPolicy
from scpn_phase_orchestrator.supervisor.policy_rules import (
    PolicyAction,
    PolicyCondition,
    PolicyRule,
)
from scpn_phase_orchestrator.supervisor.regimes import RegimeManager
from scpn_phase_orchestrator.upde.engine import UPDEEngine
from scpn_phase_orchestrator.upde.metrics import LayerState, UPDEState
from scpn_phase_orchestrator.upde.order_params import compute_order_parameter

## 1. Setup: 8-Oscillator System with Event Bus

In [ ]:
N = 8
engine = UPDEEngine(n_oscillators=N, dt=0.01, method="rk4")
event_bus = EventBus()
regime_mgr = RegimeManager(hysteresis=0.05, cooldown_steps=5, event_bus=event_bus)

boundaries = [
    BoundaryDef(
        name="r_low", variable="R_global", lower=0.3, upper=None, severity="hard"
    ),
    BoundaryDef(
        name="r_warn", variable="R_global", lower=0.6, upper=None, severity="soft"
    ),
]
observer = BoundaryObserver(boundaries)
observer.set_event_bus(event_bus)

policy = SupervisorPolicy(regime_mgr)

rng = np.random.default_rng(123)
theta = rng.uniform(0, 2 * np.pi, N)
omega = np.ones(N)
knm = np.full((N, N), 0.3)
np.fill_diagonal(knm, 0)

print(f"Initial regime: {regime_mgr.current_regime.name}")

## 2. Policy Rules (Programmatic Definition)

Two rules:
- **boost_coupling**: When `R_global < 0.5` in DEGRADED regime, increase K by 20%
- **reduce_coupling**: When `R_global > 0.9` in NOMINAL, reduce K by 10%

In [ ]:
rules = [
    PolicyRule(
        name="boost_coupling",
        regimes=["DEGRADED", "CRITICAL"],
        condition=PolicyCondition(metric="R_global", layer=None, op="<", threshold=0.5),
        actions=[PolicyAction(knob="K", scope="global", value=1.2, ttl_s=5.0)],
        cooldown_s=2.0,
    ),
    PolicyRule(
        name="reduce_coupling",
        regimes=["NOMINAL"],
        condition=PolicyCondition(metric="R_global", layer=None, op=">", threshold=0.9),
        actions=[PolicyAction(knob="K", scope="global", value=0.9, ttl_s=5.0)],
        cooldown_s=2.0,
    ),
]
print(f"Defined {len(rules)} policy rules")

## 3. Simulation Loop with Regime Tracking

In [ ]:
history_r = []
history_regime = []
alpha = np.zeros((N, N))

for step in range(500):
    # Perturbation at step 100: scramble phases
    if step == 100:
        theta = rng.uniform(0, 2 * np.pi, N)

    theta = engine.step(theta, omega, knm, 0.0, 0.0, alpha)
    R, psi = compute_order_parameter(theta)

    layer = LayerState(R=R, psi=psi, lock_signatures=[])
    upde_state = UPDEState(
        layers=[layer],
        cross_layer_alignment=np.array([[R]]),
        stability_proxy=R,
        regime_id=regime_mgr.current_regime.value,
    )

    boundary_state = observer.observe({"R_global": R}, step=step)
    actions = policy.decide(upde_state, boundary_state)

    # Apply coupling scale action
    for a in actions:
        if a.knob == "K":
            knm = knm * a.value

    history_r.append(R)
    history_regime.append(regime_mgr.current_regime.name)

print(f"Final R = {history_r[-1]:.4f}")
print(f"Final regime: {history_regime[-1]}")
print(f"Regime transitions: {len(regime_mgr.transition_history)}")

## 4. Regime Timeline

In [ ]:
try:
    import matplotlib.pyplot as plt

    regime_colors = {
        "NOMINAL": "green",
        "DEGRADED": "orange",
        "CRITICAL": "red",
        "RECOVERY": "blue",
    }
    regime_values = {"NOMINAL": 0, "DEGRADED": 1, "CRITICAL": 2, "RECOVERY": 3}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    ax1.plot(history_r, linewidth=0.8)
    ax1.axhline(y=0.3, color="red", linestyle="--", linewidth=0.5, label="CRITICAL")
    ax1.axhline(y=0.6, color="orange", linestyle="--", linewidth=0.5, label="DEGRADED")
    ax1.axvline(x=100, color="gray", linestyle=":", linewidth=0.5, label="perturbation")
    ax1.set_ylabel("R")
    ax1.set_title("Order parameter with perturbation at step 100")
    ax1.legend(fontsize=8)

    regime_nums = [regime_values[r] for r in history_regime]
    ax2.fill_between(range(len(regime_nums)), regime_nums, step="mid", alpha=0.4)
    ax2.set_yticks([0, 1, 2, 3])
    ax2.set_yticklabels(["NOMINAL", "DEGRADED", "CRITICAL", "RECOVERY"])
    ax2.set_xlabel("Step")
    ax2.set_title("Regime transitions")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed, skipping plot")

## 5. Petri Net FSM

Formal multi-place regime sequencing with guarded transitions.

In [ ]:
from scpn_phase_orchestrator.supervisor.petri_net import (
    Arc,
    Guard,
    Marking,
    PetriNet,
    Place,
    Transition,
)

# 3-place protocol: WARMUP -> ACTIVE -> COOLDOWN
places = [Place("warmup"), Place("active"), Place("cooldown")]
transitions = [
    Transition(
        name="start",
        inputs=[Arc(place="warmup", weight=1)],
        outputs=[Arc(place="active", weight=1)],
        guard=Guard(metric="R_global", op=">", threshold=0.7),
    ),
    Transition(
        name="finish",
        inputs=[Arc(place="active", weight=1)],
        outputs=[Arc(place="cooldown", weight=1)],
        guard=Guard(metric="elapsed_steps", op=">", threshold=200),
    ),
]

net = PetriNet(places=places, transitions=transitions)
marking = Marking({"warmup": 1, "active": 0, "cooldown": 0})

# Fire with R_global > 0.7: "start" transition fires
ctx = {"R_global": 0.85, "elapsed_steps": 50}
marking, fired = net.step(marking, ctx)
print(
    f"After step 1: {marking.active_places()}, fired: {fired.name if fired else None}"
)

# Fire with elapsed_steps > 200: "finish" transition fires
ctx["elapsed_steps"] = 250
marking, fired = net.step(marking, ctx)
print(
    f"After step 2: {marking.active_places()}, fired: {fired.name if fired else None}"
)

## Summary

- `PolicyRule` declares condition-action pairs gated by regime and metric thresholds
- `RegimeManager` implements NOMINAL/DEGRADED/CRITICAL/RECOVERY FSM with hysteresis
- `EventBus` propagates boundary breaches and regime transitions
- `PetriNet` provides formal multi-place protocol sequencing with guarded transitions
- All components compose into the `SupervisorPolicy.decide()` pipeline